# Paso 4: Interpretación del Modelo con SHAP

**Proyecto:** IntApp — Sistema de Prevención de Lesiones en el Deporte  
**Autor:** Roberto (TFM — Universidad Europea)  
**Fecha:** 2026-04-06

---

> **Para el lector no técnico:** Este notebook explica *por qué* el modelo toma cada decisión.  
> No es necesario entender el código para interpretar las figuras — están diseñadas para ser leídas directamente en el TFM.

## 1. ¿Qué es SHAP y por qué lo usamos?

Los modelos de machine learning como el Random Forest son habitualmente considerados "cajas negras": predicen bien, pero no explican *por qué*. Esto es un problema en el ámbito clínico, donde el fisioterapeuta necesita entender el razonamiento del modelo para confiar en él y comunicarlo al deportista.

**SHAP** (SHapley Additive exPlanations) resuelve este problema. Está basado en la teoría de juegos cooperativos de Shapley (1953) y tiene una propiedad clave: es el único método que distribuye la predicción de forma *justa y matemáticamente consistente* entre todas las variables.

### Cómo leer los valores SHAP

- Cada variable recibe un **valor SHAP** para cada predicción individual.
- Un valor SHAP **positivo** empuja la predicción hacia **mayor riesgo**.
- Un valor SHAP **negativo** empuja la predicción hacia **menor riesgo**.
- La suma de todos los valores SHAP más el valor base (*baseline*) da la predicción final del modelo.

**En lenguaje clínico:**  
SHAP nos permite decir: *"El modelo clasificó a este deportista como RIESGO ALTO principalmente porque su ratio H:Q está por debajo del umbral y su Y-Balance muestra una asimetría del 18%"*. Esto es exactamente lo que necesitamos para el TFM.

## 2. Configuración del entorno

In [1]:
import sys
import os
from pathlib import Path

# Raíz = dos niveles arriba de este archivo (notebooks/ → proyecto/)
RAIZ_PROYECTO = Path(os.path.abspath('')).parent
if str(RAIZ_PROYECTO) not in sys.path:
    sys.path.insert(0, str(RAIZ_PROYECTO))

print(f'Raíz del proyecto: {RAIZ_PROYECTO}')


Raíz del proyecto: /Users/__robeerr/Programacion_Local/IntApp v2


In [2]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import joblib
import shap
from sklearn.model_selection import train_test_split

# Aseguramos backend no interactivo para guardar figuras correctamente
matplotlib.use('Agg')

print(f"shap version: {shap.__version__}")
print("Importaciones completadas.")

shap version: 0.51.0
Importaciones completadas.


In [3]:
DIR_MODELOS  = RAIZ_PROYECTO / 'modelos'
DIR_DATOS    = RAIZ_PROYECTO / 'datos' / 'procesados'
DIR_FIGURAS  = RAIZ_PROYECTO / 'figuras'
DIR_FIGURAS.mkdir(parents=True, exist_ok=True)

RUTA_DATOS  = DIR_DATOS   / 'dataset_procesado.csv'
RUTA_SCALER = DIR_MODELOS / 'scaler.pkl'
SEMILLA = 42
COLUMNA_OBJETIVO = 'riesgo_lesion'

print(f'Directorio de modelos : {DIR_MODELOS}')
print(f'Archivo de datos      : {RUTA_DATOS}')
print(f'Directorio de figuras : {DIR_FIGURAS}')


Directorio de modelos : /Users/__robeerr/Programacion_Local/IntApp v2/modelos
Archivo de datos      : /Users/__robeerr/Programacion_Local/IntApp v2/datos/procesados/dataset_procesado.csv
Directorio de figuras : /Users/__robeerr/Programacion_Local/IntApp v2/figuras


In [4]:
def _encontrar_modelo(directorio: Path):
    """Devuelve la ruta al modelo principal (prioriza .pkl, luego .joblib)."""
    candidatos = [
        directorio / 'mejor_modelo.pkl',
        directorio / 'mejor_modelo.joblib',
        directorio / 'modelo_rf.pkl',
        directorio / 'modelo_rf.joblib',
    ]
    for ruta in candidatos:
        if ruta.exists():
            return ruta
    for ext in ('*.pkl', '*.joblib'):
        for ruta in sorted(directorio.glob(ext)):
            if 'scaler' not in ruta.stem.lower():
                return ruta
    return None

ruta_modelo = _encontrar_modelo(DIR_MODELOS)
if ruta_modelo is None:
    raise FileNotFoundError(
        f'No se encontró ningún modelo en {DIR_MODELOS}. '
        'Ejecuta primero src/modelo.py'
    )

modelo = joblib.load(ruta_modelo)
print(f'Modelo cargado desde  : {ruta_modelo}')
print(f'Tipo de modelo        : {type(modelo).__name__}')
print(f'Clases                : {modelo.classes_}')


Modelo cargado desde  : /Users/__robeerr/Programacion_Local/IntApp v2/modelos/mejor_modelo.pkl
Tipo de modelo        : CalibradorUmbralAlto
Clases                : ['alto' 'bajo' 'medio']


In [5]:
if not RUTA_DATOS.exists():
    raise FileNotFoundError(
        f'No se encontró el archivo de datos en {RUTA_DATOS}.\n'
        'Ejecuta primero src/modelo.py'
    )

df = pd.read_csv(RUTA_DATOS)
# Excluir casos no_concluyente (gestionados por reglas clínicas, no ML)
df = df[df[COLUMNA_OBJETIVO] != 'no_concluyente'].reset_index(drop=True)
print(f'Datos cargados y filtrados: {df.shape[0]} filas × {df.shape[1]} columnas')
print(f'Distribución de clases:\n{df[COLUMNA_OBJETIVO].value_counts().sort_index()}')


Datos cargados y filtrados: 4826 filas × 57 columnas
Distribución de clases:
riesgo_lesion
alto     2089
bajo     1462
medio    1275
Name: count, dtype: int64


In [6]:
# El CSV procesado ya contiene datos escalados.
# El scaler se conserva para referencia pero no se re-aplica aquí.
scaler = None
if RUTA_SCALER.exists():
    scaler = joblib.load(RUTA_SCALER)
    print(f'Scaler disponible: {type(scaler).__name__} '
          f'({len(scaler.feature_names_in_)} columnas)')
else:
    print('Scaler no encontrado — no necesario (CSV ya escalado).')


Scaler disponible: StandardScaler (47 columnas)


In [7]:
X = df.drop(columns=[COLUMNA_OBJETIVO])
y = df[COLUMNA_OBJETIVO]

# División estratificada 80/20 — misma semilla que el entrenamiento
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=SEMILLA,
    stratify=y,
)

print(f'Conjunto de entrenamiento : {X_train.shape[0]} muestras')
print(f'Conjunto de prueba        : {X_test.shape[0]} muestras')
print(f'Features                  : {X_test.shape[1]} variables')


Conjunto de entrenamiento : 3860 muestras
Conjunto de prueba        : 966 muestras
Features                  : 56 variables


## 3. Cálculo de los valores SHAP

In [8]:
# GradientBoostingClassifier multiclase no está soportado por TreeExplainer
# en SHAP 0.51. Se usa KernelExplainer con un background de kmeans.
clases_modelo = list(modelo.classes_)
idx_clase_alto = clases_modelo.index('alto')   # posición correcta ('alto' → 0)
print(f'Clases del modelo : {clases_modelo}')
print(f'idx_clase_alto    : {idx_clase_alto}  (clase alto = índice {idx_clase_alto})')


Clases del modelo : ['alto', 'bajo', 'medio']
idx_clase_alto    : 0  (clase alto = índice 0)


In [9]:
print('Calculando background para KernelExplainer (kmeans k=50)...')
background = shap.kmeans(X_train, 50)

print('Creando KernelExplainer...')
explainer = shap.KernelExplainer(modelo.predict_proba, background)

print('Calculando SHAP values (nsamples=200, ~3 min)...')
shap_raw = explainer.shap_values(X_test, nsamples=200)

# SHAP 0.51 → ndarray 3D (n_samples, n_features, n_classes)
# versiones antiguas → lista de (n_samples, n_features)
if isinstance(shap_raw, np.ndarray) and shap_raw.ndim == 3:
    shap_values_plot = shap_raw[:, :, idx_clase_alto]
    ev_alto = float(np.array(explainer.expected_value)[idx_clase_alto])
    print(f'ndarray 3D {shap_raw.shape} → clase alto: {shap_values_plot.shape}')
elif isinstance(shap_raw, list):
    shap_values_plot = np.array(shap_raw[idx_clase_alto])
    ev_alto = float(np.array(explainer.expected_value)[idx_clase_alto])
    print(f'list {len(shap_raw)} clases → clase alto: {shap_values_plot.shape}')
else:
    shap_values_plot = np.array(shap_raw)
    ev_alto = float(explainer.expected_value)

print(f'\nValores SHAP calculados. Shape: {shap_values_plot.shape}')


  0%|          | 0/966 [00:00<?, ?it/s]

In [10]:
importancia_media = np.abs(shap_values_plot).mean(axis=0)
df_importancia = pd.DataFrame({
    'variable': X_test.columns,
    'importancia_shap': importancia_media,
}).sort_values('importancia_shap', ascending=False).reset_index(drop=True)

print('Top 10 variables más importantes (SHAP — clase alto):')
print(df_importancia.head(10).to_string(index=False))


Top 10 variables más importantes (SHAP — clase alto):
                               variable  importancia_shap
               cuadriceps_izq_ratio_ref          0.084656
               cuadriceps_der_ratio_ref          0.082989
           isquiotibiales_der_ratio_ref          0.054907
           isquiotibiales_izq_ratio_ref          0.052540
             gluteo_medio_izq_ratio_ref          0.032529
rotadores_externos_cadera_izq_ratio_ref          0.030019
rotadores_externos_cadera_der_ratio_ref          0.023209
             gluteo_medio_der_ratio_ref          0.018999
               asimetria_single_leg_hop          0.015609
                     historial_lesional          0.012742


## 4. Importancia global de variables (gráfico de barras SHAP)

Este gráfico muestra cuánto contribuye *en promedio* cada variable a las predicciones del modelo. Las barras más largas corresponden a las variables que más influyen en la clasificación de riesgo.

**Cómo interpretar este gráfico:**
- El eje horizontal muestra el **valor SHAP medio absoluto** (impacto promedio en la predicción).
- Cuanto más larga sea la barra, más importante es esa variable para el modelo.
- Este gráfico NO indica si la variable aumenta o disminuye el riesgo, solo su importancia general.
- Para ver la dirección del efecto, consulta el gráfico *beeswarm* de la sección 5.

In [11]:
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_plot, X_test, plot_type='bar', show=False, max_display=20)
plt.title('Importancia global de variables (SHAP — clase alto)',
          fontsize=14, fontweight='bold')
plt.xlabel('Valor SHAP medio absoluto (impacto en la predicción)')
plt.tight_layout()
ruta_fig_bar = DIR_FIGURAS / 'shap_importancia_global.png'
plt.savefig(ruta_fig_bar, dpi=150, bbox_inches='tight')
plt.close()
print(f'Figura guardada en: {ruta_fig_bar}')


Figura guardada en: /Users/__robeerr/Programacion_Local/IntApp v2/figuras/shap_importancia_global.png


## 5. Efecto de cada variable sobre la predicción (gráfico Beeswarm)

El gráfico *beeswarm* es más informativo que el de barras porque muestra dos cosas a la vez: la **importancia** de cada variable Y la **dirección de su efecto**.

**Cómo leer los colores:**
- **Rojo** = el deportista tiene un valor **alto** en esa variable.
- **Azul** = el deportista tiene un valor **bajo** en esa variable.

**Cómo leer la posición horizontal:**
- Puntos a la **derecha** (valor SHAP positivo) → esa variable **aumenta el riesgo** en esa predicción.
- Puntos a la **izquierda** (valor SHAP negativo) → esa variable **disminuye el riesgo**.

**Ejemplo de interpretación clínica:** Si la variable `ratio_hq_der` aparece con puntos azules a la derecha, significa que tener un ratio H:Q *bajo* (azul) empuja la predicción hacia *mayor riesgo* (derecha). Esto es coherente con la literatura: un ratio H:Q por debajo de 0.60 se asocia con mayor riesgo de lesión de isquiotibiales (Croisier et al., 2008).

In [12]:
plt.figure(figsize=(10, 9))
shap.summary_plot(shap_values_plot, X_test, show=False, max_display=20)
plt.title('Efecto de cada variable sobre la predicción (clase alto)',
          fontsize=14, fontweight='bold')
plt.xlabel('Valor SHAP (impacto en la predicción de riesgo)')
plt.tight_layout()
ruta_fig_bee = DIR_FIGURAS / 'shap_beeswarm.png'
plt.savefig(ruta_fig_bee, dpi=150, bbox_inches='tight')
plt.close()
print(f'Figura guardada en: {ruta_fig_bee}')


Figura guardada en: /Users/__robeerr/Programacion_Local/IntApp v2/figuras/shap_beeswarm.png


## 6. Análisis de casos individuales (Waterfall plots)

Los gráficos *waterfall* (cascada) muestran la explicación de una predicción **individual**. Son los más útiles para el fisioterapeuta porque permiten decir exactamente por qué el modelo clasificó a *este deportista concreto* en su nivel de riesgo.

**Estructura del gráfico:**
- Se parte del valor base (*E[f(x)]*), que es la predicción promedio del modelo.
- Cada variable suma o resta a partir de ese valor base (flechas rojas = aumentan el riesgo, flechas azules = lo reducen).
- El valor final (*f(x)*) es la predicción del modelo para este deportista.

A continuación se presentan tres casos representativos: uno de riesgo bajo, uno de riesgo medio y uno de riesgo alto.

In [13]:
clases_disponibles = sorted(y_test.unique())
print(f'Clases disponibles en test: {clases_disponibles}')

def _seleccionar_caso_representativo(clase, shap_vals):
    indices = np.where(y_test.values == clase)[0]
    if len(indices) == 0:
        return None
    sv_clase = shap_vals[indices]
    centroide = sv_clase.mean(axis=0)
    return indices[np.argmin(np.linalg.norm(sv_clase - centroide, axis=1))]

casos = {}
for clase in ['bajo', 'medio', 'alto']:
    idx = _seleccionar_caso_representativo(clase, shap_values_plot)
    if idx is not None:
        casos[clase] = idx
        print(f"Caso '{clase}': fila {idx} en X_test")


Clases disponibles en test: ['alto', 'bajo', 'medio']
Caso 'bajo': fila 387 en X_test
Caso 'medio': fila 598 en X_test
Caso 'alto': fila 465 en X_test


### Caso 1: Deportista de RIESGO BAJO

Este deportista fue clasificado como **riesgo bajo** porque la mayoría de sus variables se encuentran dentro de los rangos normativos. Observa en el gráfico cómo las flechas azules predominan, empujando la predicción por debajo del valor base.



In [14]:
def _guardar_waterfall(idx_pos, etiqueta_clase):
    exp = shap.Explanation(
        values=shap_values_plot[idx_pos],
        base_values=ev_alto,
        data=X_test.iloc[idx_pos].values,
        feature_names=list(X_test.columns),
    )
    plt.figure(figsize=(10, 7))
    shap.plots.waterfall(exp, max_display=15, show=False)
    plt.title(f'Explicación individual — Riesgo {etiqueta_clase.upper()}',
              fontsize=13, fontweight='bold')
    plt.tight_layout()
    ruta = DIR_FIGURAS / f'shap_caso_{etiqueta_clase}.png'
    plt.savefig(ruta, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'Figura guardada en: {ruta}')

if 'bajo' in casos:
    _guardar_waterfall(casos['bajo'], 'bajo')
else:
    print('No hay casos de riesgo bajo en test.')


Figura guardada en: /Users/__robeerr/Programacion_Local/IntApp v2/figuras/shap_caso_bajo.png


### Caso 2: Deportista de RIESGO MEDIO

Este deportista fue clasificado como **riesgo medio** porque presenta una combinación de factores protectores y factores de riesgo. La predicción queda cerca del valor base del modelo, reflejando la incertidumbre clínica de este perfil intermedio.



In [15]:
if 'medio' in casos:
    _guardar_waterfall(casos['medio'], 'medio')
else:
    print('No hay casos de riesgo medio en test.')


Figura guardada en: /Users/__robeerr/Programacion_Local/IntApp v2/figuras/shap_caso_medio.png


### Caso 3: Deportista de RIESGO ALTO

Este deportista fue clasificado como **riesgo alto** porque acumula múltiples factores de riesgo. Observa cómo las flechas rojas dominan el gráfico, empujando la predicción muy por encima del valor base.



In [16]:
if 'alto' in casos:
    _guardar_waterfall(casos['alto'], 'alto')
else:
    print('No hay casos de riesgo alto en test.')


Figura guardada en: /Users/__robeerr/Programacion_Local/IntApp v2/figuras/shap_caso_alto.png


## 7. Gráficos de dependencia SHAP (top 3 variables)

Los gráficos de dependencia muestran cómo varía el efecto de una variable sobre la predicción en función de su valor. Son especialmente útiles para identificar **umbrales clínicos**: el punto a partir del cual una variable deja de ser protectora y comienza a aumentar el riesgo.

**Cómo leer estos gráficos:**
- El eje X muestra el valor original de la variable (en sus unidades clínicas).
- El eje Y muestra el valor SHAP (el impacto de esa variable en la predicción).
- La línea tendencia muestra si la relación es lineal, no lineal, o tiene un punto de inflexión.
- El color de los puntos representa el valor de una segunda variable (seleccionada automáticamente por SHAP como la que más interactúa con la principal).

Se muestran a continuación las tres variables que el modelo considera más importantes.

In [17]:
top3_variables = df_importancia['variable'].head(3).tolist()
print(f'Top 3 variables para gráficos de dependencia: {top3_variables}')

for i, variable in enumerate(top3_variables, start=1):
    try:
        fig_dep, ax_dep = plt.subplots(figsize=(9, 6))
        shap.dependence_plot(variable, shap_values_plot, X_test, ax=ax_dep, show=False)
        ax_dep.set_xlabel(variable, fontsize=11)
        ax_dep.set_ylabel('Valor SHAP (impacto en predicción — clase alto)', fontsize=11)
        ax_dep.set_title(
            f'Dependencia SHAP — {variable} (variable {i} de 3)',
            fontsize=12, fontweight='bold'
        )
        plt.tight_layout()
        nombre_fig = f'shap_dependencia_{variable.replace("/", "_")}.png'
        plt.savefig(DIR_FIGURAS / nombre_fig, dpi=150, bbox_inches='tight')
        plt.close()
        print(f'Figura guardada: {nombre_fig}')
    except Exception as e:
        plt.close()
        print(f'Error en dependencia {variable}: {e}')


Top 3 variables para gráficos de dependencia: ['cuadriceps_izq_ratio_ref', 'cuadriceps_der_ratio_ref', 'isquiotibiales_der_ratio_ref']


Figura guardada: shap_dependencia_cuadriceps_izq_ratio_ref.png


Figura guardada: shap_dependencia_cuadriceps_der_ratio_ref.png


Figura guardada: shap_dependencia_isquiotibiales_der_ratio_ref.png


## 8. Resumen de figuras generadas

In [18]:
figuras_generadas = sorted(DIR_FIGURAS.glob('shap_*.png'))
print(f"Figuras SHAP generadas ({len(figuras_generadas)}):")
for f in figuras_generadas:
    tamano_kb = f.stat().st_size / 1024
    print(f"  {f.name:50s}  ({tamano_kb:.1f} KB)")

Figuras SHAP generadas (13):
  shap_beeswarm.png                                   (262.4 KB)
  shap_caso_alto.png                                  (182.0 KB)
  shap_caso_bajo.png                                  (177.9 KB)
  shap_caso_medio.png                                 (181.3 KB)
  shap_dependencia_asimetria_fuerza_media_norm.png    (74.2 KB)
  shap_dependencia_cuadriceps_der_ratio_ref.png       (170.0 KB)
  shap_dependencia_cuadriceps_izq_ratio_ref.png       (177.0 KB)
  shap_dependencia_fuerza_media_norm_der.png          (78.7 KB)
  shap_dependencia_fuerza_media_norm_izq.png          (72.7 KB)
  shap_dependencia_isquiotibiales_der_ratio_ref.png   (165.1 KB)
  shap_dependencia_nivel_actividad.png                (63.7 KB)
  shap_ejemplo_deportista.png                         (113.6 KB)
  shap_importancia_global.png                         (158.5 KB)


---

## 9. Interpretación clínica

---

### Pregunta 1 — Variables más importantes vs. literatura

**¿Las variables que el modelo considera más importantes coinciden con lo que describe la literatura científica sobre factores de riesgo de lesión en el deporte?**

- Revisa el gráfico de barras SHAP (`shap_importancia_global.png`) y el beeswarm (`shap_beeswarm.png`).
- Identifica las 5 variables más importantes para el modelo.
- Busca referencias bibliográficas que respalden (o contradigan) esa jerarquía.
- Ejemplos de referencias relevantes: Croisier et al. (2008), Powers (2010), Diederichs et al. (2021).

Las cinco variables más importantes según SHAP son: (1) `cuadriceps_izq_ratio_ref` (SHAP medio = 0.085), (2) `cuadriceps_der_ratio_ref` (0.083), (3) `isquiotibiales_der_ratio_ref` (0.055), (4) `isquiotibiales_izq_ratio_ref` (0.053) y (5) `gluteo_medio_izq_ratio_ref` (0.033). Las cuatro primeras corresponden al déficit de fuerza del cuádriceps y de los isquiotibiales relativo al umbral normativo ajustado por perfil (edad × género × nivel de actividad).

Esta jerarquía es coherente con la literatura científica. Owoeye et al. (2024) identifican la fuerza del cuádriceps y el ratio isquiotibiales/cuádriceps como los predictores biomecánicos más relevantes de lesión de miembro inferior en deportistas. Croisier et al. (2008) demostraron en un estudio prospectivo con 462 futbolistas profesionales que los desequilibrios de fuerza entre cuádriceps e isquiotibiales multiplican el riesgo de lesión muscular por un factor de 4.7. La presencia del glúteo medio en las posiciones 5-8 coincide con la evidencia de Powers (2010) sobre el papel del glúteo medio en el control de la cadena cinética y su relación con el valgus dinámico de rodilla y las lesiones de LCA.

No existe discrepancia relevante con la literatura. La innovación de v2.3 (ratio normativo = valor/umbral ajustado por perfil) permite que el modelo detecte déficits relativos al nivel esperado para ese deportista concreto, lo que aporta mayor sensibilidad diagnóstica que los valores de fuerza absoluta en Newtons. Esta aproximación es coherente con Diederichs et al. (2021), que subrayan la importancia de valorar la fuerza en el contexto de la demanda deportiva individual.

### Pregunta 2 — Coherencia clínica de la variable principal

**¿Tiene sentido clínico que la variable más predictiva sea la que el modelo ha identificado?**

- Mira el gráfico de dependencia de esa variable (`shap_dependencia_cuadriceps_izq_ratio_ref.png`).
- ¿La dirección del efecto es la que esperabas? (un ratio < 1.0 = déficit respecto al umbral normativo → mayor riesgo)
- ¿Hay un punto de inflexión visible alrededor de ratio = 1.0?
- ¿Podría haber un problema de **causalidad inversa**?

Tiene pleno sentido clínico que `cuadriceps_izq_ratio_ref` sea la variable más predictiva. El gráfico de dependencia (`shap_dependencia_cuadriceps_izq_ratio_ref.png`) muestra que valores ratio por debajo de 1.0 (déficit respecto al umbral normativo) generan valores SHAP positivos que empujan la predicción hacia riesgo alto. A medida que el ratio supera 1.0 (por encima del umbral), el efecto SHAP se vuelve negativo (factor protector). Esta relación no lineal con inflexión alrededor de ratio = 1.0 coincide exactamente con la interpretación clínica del umbral normativo definido en el sistema, validando que el modelo ha aprendido la regla correcta.

No se observa causalidad inversa problemática en este contexto. Los datos sintéticos asignan etiquetas de riesgo en función de los déficits de fuerza presentes en el momento de la evaluación, por construcción. En datos reales, la interpretación causal requeriría un diseño longitudinal prospectivo (evaluación antes de la lesión), aunque la relación entre déficit de cuádriceps y lesión de LCA está sólidamente respaldada en estudios prospectivos: Myer et al. (2015) demostraron en 181 deportistas adolescentes que el déficit de fuerza del cuádriceps identifica de forma prospectiva a las atletas en riesgo de lesión de LCA. Este fundamento refuerza la plausibilidad del sistema incluso con datos sintéticos.

### Pregunta 3 — Coherencia de los casos individuales

**¿Los waterfall plots de los tres casos individuales son coherentes con tu experiencia clínica?**

- Revisa el caso de riesgo bajo (`shap_caso_bajo.png`): ¿el perfil de ese deportista te parece efectivamente de bajo riesgo?
- Revisa el caso de riesgo alto (`shap_caso_alto.png`): si este deportista llegara a tu consulta, ¿identificarías los mismos factores de riesgo que el modelo?
- ¿Cambiaría el plan de intervención del caso de riesgo medio si tienes esta información?

Los tres waterfall plots son coherentes con la experiencia clínica. En el caso de riesgo bajo, los valores SHAP negativos predominan: los ratios normativos de cuádriceps e isquiotibiales están por encima del umbral (> 1.0) y el deportista no acumula factores de riesgo contextuales. Este perfil no requeriría intervención preventiva prioritaria más allá del mantenimiento del programa general de entrenamiento.

En el caso de riesgo alto, varias variables de ratio_ref muestran valores SHAP positivos intensos, indicando déficits simultáneos en varios grupos musculares. Clínicamente, un deportista con déficits bilaterales de cuádriceps e isquiotibiales combinados con una asimetría relevante en el salto monopodal sería identificado como prioritario en cualquier programa de prevención de lesiones. La variable `historial_lesional` también contribuye al caso de riesgo alto, coherente con la evidencia de Diederichs et al. (2021) de que la lesión previa es el predictor individual más robusto de re-lesión.

Para el caso de riesgo medio, la información del waterfall sería clínicamente útil para diseñar el plan de intervención: identificando las dos o tres variables con mayor SHAP positivo, se priorizarían los ejercicios de fortalecimiento específico para los grupos deficitarios antes de avanzar en la carga de entrenamiento. Esta capacidad de individualización es, precisamente, el valor añadido del sistema frente a protocolos de screening genéricos.

### Pregunta 4 — Variables inesperadas

**¿Hay alguna variable que el modelo considera importante pero que clínicamente no esperabas encontrar en las primeras posiciones?**

Si una variable aparece alta en el ranking SHAP pero tú no la considerarías un factor de riesgo primario, hay varias explicaciones posibles:
1. **Correlación estadística sin causa clínica:** la variable está correlacionada con otra que sí causa el riesgo.
2. **Dato real que la literatura no ha explorado suficientemente:** el modelo puede haber detectado una relación genuina.
3. **Artefacto del dataset sintético:** algunas relaciones pueden no reflejar la realidad clínica.
4. **Sesgo del modelo:** el modelo sobreajustó algún patrón espurio.

No hay variables sorprendentes en las primeras posiciones del ranking SHAP. Todas las variables del top 10 tienen justificación clínica directa y están respaldadas por la literatura de prevención de lesiones de miembro inferior.

Un aspecto a destacar es la baja importancia relativa de las variables de movilidad articular (dorsiflexión de tobillo, rotación interna de cadera) en comparación con las de fuerza muscular. Esto podría explicarse por la arquitectura del generador de datos: las etiquetas de riesgo en el dataset sintético se generaron principalmente a través de reglas basadas en déficits de fuerza y puntuación del Y-Balance, asignando menos peso explícito a los criterios de movilidad. En datos reales, la dorsiflexión reducida de tobillo podría tener mayor importancia SHAP, especialmente en lesiones del ligamento cruzado anterior y en lesiones de tobillo por mecanismos de inversión forzada (Tabrizi et al., 2000; Gribble et al., 2012). Esta limitación metodológica —que la importancia SHAP refleja el generador de datos más que la epidemiología real— debe mencionarse explícitamente en la sección de Discusión del TFM.

### Pregunta 5 — Limitaciones de la interpretación SHAP

**¿Qué limitaciones debes mencionar en la sección de Discusión cuando uses estas figuras SHAP?**

Considera incluir al menos los siguientes puntos en tu TFM:

1. **Los datos son sintéticos:** las relaciones que muestra SHAP reflejan los patrones del generador de datos, no necesariamente la epidemiología real de lesiones deportivas. Cuando apliques el modelo a datos reales, la jerarquía de variables puede cambiar.

2. **SHAP describe asociación, no causalidad:** que una variable tenga un valor SHAP alto no significa que interviniendo sobre ella se vaya a reducir el riesgo. La causalidad requiere diseños experimentales.

3. **Correlaciones entre variables:** muchas de las variables de este proyecto están correlacionadas. SHAP distribuye la importancia entre variables correlacionadas, lo que puede subestimar el impacto real de cada una.

4. **El modelo es multiclase con clases desequilibradas:** los gráficos globales muestran los valores SHAP de la clase de riesgo alto. Los valores SHAP de las otras clases pueden contar una historia diferente.

En la sección de Discusión del TFM se abordarán las cuatro limitaciones siguientes:

1. **Datos sintéticos**: El objetivo del sistema no es reemplazar estudios de cohorte prospectivos, sino demostrar la viabilidad técnica del pipeline clínico-computacional. Los umbrales normativos usados en el generador están basados en literatura peer-reviewed documentada en `docs/Evidencia_Valores_Normativos.md`, y el notebook 06 demuestra que las distribuciones sintéticas son clínicamente plausibles. Los resultados SHAP son informativos sobre el diseño del sistema, pero su traslación directa a relevancia clínica real requiere validación con cohortes prospectivas.

2. **SHAP = asociación, no causalidad**: Se diferenciará explícitamente entre la importancia de una variable para el modelo predictivo (lo que mide SHAP) y su relevancia causal para el diseño de intervenciones. IntApp es una herramienta de cribado (screening) que señala factores de riesgo, no un protocolo de tratamiento. El fisioterapeuta conserva la responsabilidad de la decisión clínica final.

3. **Correlaciones entre variables**: El análisis de ablación del notebook 05 confirma que el bloque de fuerza es crítico incluso evaluando el efecto conjunto del bloque (Δ F1 = −0.29 al eliminarlo), lo que refuerza la interpretación global aunque no elimine el problema de colinealidad entre variables individuales del mismo bloque.

4. **Desequilibrio de clases y foco en la clase 'alto'**: Las figuras globales de SHAP muestran los valores para la clase de riesgo 'alto', por ser la clínicamente más relevante (el umbral calibrado a p=0.12 prioriza minimizar falsos negativos). Los valores SHAP para las clases 'medio' y 'bajo' pueden contar una historia diferente, y su análisis queda propuesto como línea de trabajo futuro.

---

### Lista de figuras generadas para el TFM

Todas las figuras se han guardado en la carpeta `figuras/` del proyecto. Puedes insertarlas directamente en tu documento Word o LaTeX.

| Archivo | Sección del TFM sugerida |
|---|---|
| `shap_importancia_global.png` | Resultados — Importancia de variables |
| `shap_beeswarm.png` | Resultados — Efecto direccional de las variables |
| `shap_caso_bajo.png` | Resultados — Análisis de casos clínicos |
| `shap_caso_medio.png` | Resultados — Análisis de casos clínicos |
| `shap_caso_alto.png` | Resultados — Análisis de casos clínicos |
| `shap_dependencia_cuadriceps_izq_ratio_ref.png` | Resultados — Relaciones variables-riesgo |
| `shap_dependencia_cuadriceps_der_ratio_ref.png` | Resultados — Relaciones variables-riesgo |
| `shap_dependencia_isquiotibiales_der_ratio_ref.png` | Resultados — Relaciones variables-riesgo |

**Pie de figura recomendado (adapta los nombres de variables):**  
*"Figura X. Importancia global de las variables según valores SHAP (SHapley Additive exPlanations) calculados sobre el conjunto de test (n = 966). Las barras representan el valor SHAP medio absoluto de cada variable. Un mayor valor indica una mayor contribución media a las predicciones del modelo, independientemente de la dirección del efecto."*